In [11]:
import requests as rq
from bs4 import BeautifulSoup as bs 
from bs4 import Comment
from random_user_agent.user_agent import UserAgent
from random_user_agent.params import SoftwareName, OperatingSystem
import time
import pandas as pd
import numpy as np

In [2]:
software_names = [SoftwareName.CHROME.value]
operating_systems = [OperatingSystem.WINDOWS.value, OperatingSystem.LINUX.value]   
user_agent_rotator = UserAgent(software_names=software_names, operating_systems=operating_systems, limit=100)
user_agents = user_agent_rotator.get_user_agents()
user_agent = user_agent_rotator.get_random_user_agent()


In [3]:
header = {"User-Agent" : user_agent , "Accept-Language": "en-US"}

In [6]:
page_source = rq.get('https://www.basketball-reference.com/teams/', headers = header)
soup = bs(page_source.text, 'html.parser')

teams_href = [a['href']for a in soup.select('tr.full_table a[href^="/teams/"]')]

In [7]:
def get_team_data(url ) :
    
    
    #gets the data for the franchise
    team_page_source = rq.get( 'https://www.basketball-reference.com' + url ,headers = header)
    team_soup = bs(team_page_source.text , 'html.parser')
    
    #the franchise name using slicing 
    franchise_name = team_soup.select_one('h1').text
    franchise_name = franchise_name[1:-2] 
    
    #location of the franchise using next sibling
    loc_str = team_soup.find('strong', string='Location:')
    location = loc_str.next_sibling.strip()
    
    #list of all the previous names and the current name of the franshise
    names = []
    name_str = team_soup.find('strong', string='Team Names:')
    if name_str :
        names_list = name_str.next_sibling.strip()
        for name in names_list.split(','):
            names.append(name)
    else :
        name = [np.nan]
    #seasons played datas(number of seasons , the period time of seasons played(using slicing))
    seasons_str = team_soup.find('strong', string='Seasons:') 
    seasons_info = seasons_str.next_sibling.strip().split(';')
    seasons_played = seasons_info[0].split('(')[0].strip()
    seasons_played_FT = seasons_info[1][6:] 

    #records data(total wins, total loss, WL ratio)
    record_str = team_soup.find('strong', string='Record:') 
    record_info = record_str.next_sibling.strip().split(',')
    win_loss = record_info[0].split('-')
    total_matches = int(win_loss[0]) + int(win_loss[1])
    total_wins = int(win_loss[0])
    total_loss = int(win_loss[1])
    w_l_ratio = record_info[1][2:].split('\n')[0]

    #number of playoff apearances
    PA_str = team_soup.find('strong', string='Playoff Appearances:') 
    playoff_appearances = PA_str.next_sibling.strip().split('\n')[0]
    
    #number of championships won 
    Champ_str = team_soup.find('strong', string='Championships:') 
    Championships = Champ_str.next_sibling.strip().split('\n')[0]

    #name of the current coach of the franchise
    coach = team_soup.select_one('td[data-stat="coaches"] a').text

    #league or leagues the franshise has played in 
    league = [a.text for a in team_soup.select('td[data-stat="lg_id"] a')]
    leageus = list(set(league))

    #number of hall of famers
    HOF_href = team_soup.find('a', string='Hall of Fame')['href']
    HOF_page_source = rq.get('https://www.basketball-reference.com' + HOF_href , headers = header)
    HOF_soup = bs(HOF_page_source.text , 'html.parser')
    HOF_text = HOF_soup.select_one('h2').text
    if 'More' in HOF_text :
        HOF_count = 0
    else :
        HOF_count = int(HOF_text.split('Hall of Famer')[0].strip())

    #number of all stars games players selected from the franshise 
    ASG_href = team_soup.find('a', string='All-Star Game')['href']
    ASG_page_source = rq.get('https://www.basketball-reference.com' + ASG_href , headers = header)
    ASG_soup = bs(ASG_page_source.text , 'html.parser')
    ASG_text = ASG_soup.select_one('h2').text
    if 'More' in ASG_text :
        ASG_count = 0 
    else : 
        ASG_count = int(ASG_text.split('Selections')[0].strip())
    
    #name of the franchise arena
    team_info_link = [a['href'] for a in team_soup.select('th[scope="row"] a')]

    for link in team_info_link : 
        team_info_source = rq.get('https://www.basketball-reference.com' + link , headers = header)
        team_info_soup = bs(team_info_source.text , 'html.parser')
        if team_info_soup.find('strong' , string = 'Arena:') :
            arena_name = team_info_soup.find('strong' , string = 'Arena:').next_sibling.strip()
            break
    '''
    print(f'franshise name = {franchise_name}\n')
    print(f'location = {location} \n')
    print(f'club names = {names} \n')
    print(f'seasons played = {seasons_played} \n')
    print(f'seasons played(period) = {seasons_played_FT} \n') 
    print(f'total matches = {total_matches} \n')
    print(f'total wins = {total_wins} \n ')
    print(f'total loss = {total_loss} \n ')
    print(f'WL ratio = {w_l_ratio} \n ')
    print(f'playoff appearances = {playoff_appearances} \n')
    print(f'Championships = {Championships} \n' )
    print(f'coach = {coach} \n ')
    print(f'leageus = {leageus} \n') 
    print(f'hall of famers = {HOF_count} \n')
    print(f'all star gamers = {ASG_count} \n') 
    print(f'arena name = {arena_name} \n\n ')
    '''
    return franchise_name , location , names , seasons_played , seasons_played_FT , total_matches , total_wins , total_loss , w_l_ratio , playoff_appearances , Championships , coach , leageus , HOF_count , ASG_count , arena_name

    

In [8]:
df = pd.DataFrame(columns = ['franchise_name' , 'location' , 'names' , 'seasons_played' ,
                             'seasons_played_FT' , 'total_matches' , 'total_wins' , 'total_loss' ,
                             'w_l_ratio' , 'playoff_appearances' , 'Championships' , 'coach' , 'leageus' ,
                             'HOF_count' , 'ASG_count' , 'arena_name'])

In [12]:

row_number = 1

for team in teams_href :
    franchise_name , location , names , seasons_played , seasons_played_FT , total_matches , total_wins , total_loss , w_l_ratio , playoff_appearances , Championships , coach , leageus , HOF_count , ASG_count , arena_name = get_team_data(team)
    row = {'franchise_name' : franchise_name ,
    'location' : location , 
    'names' : names ,
    'seasons_played' : seasons_played , 
    'seasons_played_FT' : seasons_played_FT , 
    'total_matches' : total_matches , 
    'total_wins' : total_wins ,
    'total_loss' : total_loss ,
    'w_l_ratio' : w_l_ratio , 
    'playoff_appearances' : playoff_appearances ,
    'Championships' : Championships , 
    'coach' : coach , 
    'leageus' : leageus  , 
    'HOF_count' : HOF_count , 
    'ASG_count' : ASG_count , 
    'arena_name' : arena_name} 
    time.sleep(5)
    df.loc[len(df)] = row 
    print(f'row {row_number} done!')
    row_number = row_number + 1
        

row 1 done!
row 2 done!


KeyboardInterrupt: 

In [150]:
df.to_excel('tems.xlsx' , index=False)